# Reproduce verification and inference

Cleaned workflow, not the notebook that generated the recorded scores. No GPU replay was performed during packaging. In Colab upload the repository ZIP below; locally start in the repository root. First run the CPU-only evidence verification. GPU inference downloads the pinned base model. Training is optional and creates a new experiment.

In [ ]:
from pathlib import Path
import os, zipfile
if not Path('scripts/verify_results.py').exists():
    from google.colab import files
    uploaded = files.upload()  # select household-preference-lora-github.zip
    assert len(uploaded) == 1
    import io
    with zipfile.ZipFile(io.BytesIO(next(iter(uploaded.values())))) as z:
        dest = Path('/content/repo-unpacked').resolve()
        for name in z.namelist():
            assert (dest / name).resolve().is_relative_to(dest)
        z.extractall(dest)
    os.chdir(dest / 'household-preference-lora')
print(Path.cwd())

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/verify_results.py"], check=True)

## Inference dependencies
Use a GPU runtime. Keep a compatible Colab PyTorch installation. Install pinned dependencies below and restart the session if Python requests it. After restarting, rerun the setup cell to enter the repository. Do not execute the historical notebook installation/recovery sequence.

In [ ]:
%pip install -r requirements.txt

If an old optional TorchAO installation causes a PEFT import error, remove it in this dedicated runtime (`%pip uninstall -y torchao`) and restart. This experiment does not require TorchAO.

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "scripts/evaluate.py", "--output", "outputs/final-replay"], check=True)

## Optional new training run
The following command is intentionally commented out. It starts a fresh base, wraps once, trains two stages, and writes separate artifacts. Default FP32 differs from the original recorded BF16 setting. Evaluate new runs on validation first; published test cases are exposed.

In [ ]:
# subprocess.run([sys.executable, "scripts/train.py", "--precision", "fp32", "--output", "outputs/new-training"], check=True)